In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType


# ============================================================
# BRONZE -> SILVER TRANSFORMATION
# ============================================================


# ============================================================
# 1. CONFIGURATION
# ============================================================

SOURCE_TABLE = "ujjivan_2.bronze.bank_transaction_fraud_detection"

TARGET_TABLE = "ujjivan_2.silver.bank_transaction_fraud_detection"

# Explicit S3 location for Silver
SILVER_PATH = "s3://ujjivanpoc/Silver/bank_transaction_fraud_detection"


# ============================================================
# 2. READ BRONZE DATA
# ============================================================

print("==============================================")
print("READING BRONZE DATA")
print("==============================================")

bronze_df = spark.table(SOURCE_TABLE)

bronze_count = bronze_df.count()

print("Bronze record count:", bronze_count)


# ============================================================
# 3. CLEAN AND TRANSFORM DATA
# ============================================================

print("\n==============================================")
print("CLEANING AND TRANSFORMING DATA")
print("==============================================")


silver_df = (

    bronze_df

    # --------------------------------------------------------
    # Customer Information
    # --------------------------------------------------------

    .withColumn(
        "Customer_ID",
        F.trim(F.col("Customer_ID"))
    )

    .withColumn(
        "Customer_Name",
        F.trim(F.col("Customer_Name"))
    )

    .withColumn(
        "Gender",
        F.initcap(F.trim(F.col("Gender")))
    )

    .withColumn(
        "Age",
        F.col("Age").cast(IntegerType())
    )

    .withColumn(
        "State",
        F.initcap(F.trim(F.col("State")))
    )

    .withColumn(
        "City",
        F.initcap(F.trim(F.col("City")))
    )


    # --------------------------------------------------------
    # Branch / Account
    # --------------------------------------------------------

    .withColumn(
        "Bank_Branch",
        F.trim(F.col("Bank_Branch"))
    )

    .withColumn(
        "Account_Type",
        F.initcap(F.trim(F.col("Account_Type")))
    )


    # --------------------------------------------------------
    # Transaction Information
    # --------------------------------------------------------

    .withColumn(
        "Transaction_ID",
        F.trim(F.col("Transaction_ID"))
    )


    # --------------------------------------------------------
    # Transaction Date
    #
    # Expected Bronze format:
    # 2025-01-23
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Date",
        F.to_date(
            F.col("Transaction_Date").cast("string"),
            "yyyy-MM-dd"
        )
    )


    # --------------------------------------------------------
    # Transaction Time
    #
    # Output:
    # HH:mm:ss
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Time",
        F.date_format(
            F.to_timestamp(
                F.col("Transaction_Time").cast("string")
            ),
            "HH:mm:ss"
        )
    )


    # --------------------------------------------------------
    # Transaction Amount
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Amount",
        F.col("Transaction_Amount")
        .cast(DecimalType(18, 2))
    )


    # --------------------------------------------------------
    # Merchant Information
    # --------------------------------------------------------

    .withColumn(
        "Merchant_ID",
        F.trim(F.col("Merchant_ID"))
    )

    .withColumn(
        "Transaction_Type",
        F.initcap(F.trim(F.col("Transaction_Type")))
    )

    .withColumn(
        "Merchant_Category",
        F.initcap(F.trim(F.col("Merchant_Category")))
    )


    # --------------------------------------------------------
    # Account Balance
    # --------------------------------------------------------

    .withColumn(
        "Account_Balance",
        F.col("Account_Balance")
        .cast(DecimalType(18, 2))
    )


    # --------------------------------------------------------
    # Device Information
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Device",
        F.initcap(F.trim(F.col("Transaction_Device")))
    )

    .withColumn(
        "Transaction_Location",
        F.trim(F.col("Transaction_Location"))
    )

    .withColumn(
        "Device_Type",
        F.upper(F.trim(F.col("Device_Type")))
    )


    # --------------------------------------------------------
    # Fraud
    # --------------------------------------------------------

    .withColumn(
        "Is_Fraud",
        F.col("Is_Fraud").cast(IntegerType())
    )


    # --------------------------------------------------------
    # Currency
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Currency",
        F.upper(F.trim(F.col("Transaction_Currency")))
    )


    # --------------------------------------------------------
    # Customer Contact
    # --------------------------------------------------------

    .withColumn(
        "Customer_Contact",
        F.trim(F.col("Customer_Contact"))
    )


    # --------------------------------------------------------
    # Transaction Description
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Description",
        F.trim(F.col("Transaction_Description"))
    )


    # --------------------------------------------------------
    # Customer Email
    # --------------------------------------------------------

    .withColumn(
        "Customer_Email",
        F.lower(F.trim(F.col("Customer_Email")))
    )
)


# ============================================================
# 4. CREATE DISTRICT COLUMN
# ============================================================

print("\n==============================================")
print("CREATING DISTRICT COLUMN")
print("==============================================")


# Example:
#
# Chennai, Tamil Nadu
#        |
#        +----> Tamil Nadu
#
# Thiruvananthapuram, Kerala
#        |
#        +----> Kerala


silver_df = (
    silver_df
    .withColumn(
        "District",
        F.when(
            F.col("Transaction_Location").contains(","),
            F.trim(
                F.element_at(
                    F.split(
                        F.col("Transaction_Location"),
                        ","
                    ),
                    -1
                )
            )
        ).otherwise(
            F.lit(None).cast("string")
        )
    )
)


# ============================================================
# 5. DATA QUALITY FILTER
# ============================================================

print("\n==============================================")
print("APPLYING DATA QUALITY FILTER")
print("==============================================")


silver_df = (
    silver_df
    .filter(
        F.col("Customer_ID").isNotNull()
        &
        F.col("Transaction_ID").isNotNull()
        &
        F.col("Transaction_Date").isNotNull()
    )
)


after_quality_count = silver_df.count()

print(
    "Records after data quality filter:",
    after_quality_count
)


# ============================================================
# 6. REMOVE DUPLICATE TRANSACTIONS
# ============================================================

print("\n==============================================")
print("REMOVING DUPLICATE TRANSACTIONS")
print("==============================================")


before_duplicate_count = silver_df.count()


silver_df = (
    silver_df
    .dropDuplicates(
        ["Transaction_ID"]
    )
)


after_duplicate_count = silver_df.count()


duplicates_removed = (
    before_duplicate_count
    -
    after_duplicate_count
)


print(
    "Records before duplicate removal:",
    before_duplicate_count
)

print(
    "Records after duplicate removal:",
    after_duplicate_count
)

print(
    "Duplicate records removed:",
    duplicates_removed
)


# ============================================================
# 7. ADD AUDIT COLUMNS
# ============================================================

print("\n==============================================")
print("ADDING AUDIT COLUMNS")
print("==============================================")


silver_df = (
    silver_df

    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_source_layer",
        F.lit("bronze")
    )
)


# ============================================================
# 8. SELECT FINAL SILVER COLUMNS
# ============================================================

print("\n==============================================")
print("PREPARING FINAL SILVER DATASET")
print("==============================================")


silver_df = silver_df.select(

    # Customer
    "Customer_ID",
    "Customer_Name",
    "Gender",
    "Age",
    "State",
    "City",

    # Branch / Account
    "Bank_Branch",
    "Account_Type",

    # Transaction
    "Transaction_ID",
    "Transaction_Date",
    "Transaction_Time",
    "Transaction_Amount",

    # Merchant
    "Merchant_ID",
    "Transaction_Type",
    "Merchant_Category",

    # Account
    "Account_Balance",

    # Device / Location
    "Transaction_Device",
    "Transaction_Location",
    "District",
    "Device_Type",

    # Fraud
    "Is_Fraud",

    # Currency
    "Transaction_Currency",

    # Customer contact
    "Customer_Contact",

    # Description
    "Transaction_Description",

    # Email
    "Customer_Email",

    # Auto Loader metadata
    "_source_file",
    "_ingested_at",
    "_rescued_data",

    # Silver audit
    "_silver_processed_timestamp",
    "_source_layer"
)


# ============================================================
# 9. DISPLAY SILVER DATA BEFORE WRITING
# ============================================================

print("\n==============================================")
print("SILVER DATA PREVIEW")
print("==============================================")


display(
    silver_df.limit(20)
)


# ============================================================
# 10. SILVER RECORD COUNT
# ============================================================

silver_dataframe_count = silver_df.count()

print(
    "\nSilver DataFrame record count:",
    silver_dataframe_count
)


# ============================================================
# 11. CHECK TARGET TABLE
# ============================================================

print("\n==============================================")
print("CHECKING SILVER TARGET TABLE")
print("==============================================")


table_exists = spark.catalog.tableExists(TARGET_TABLE)


if table_exists:

    print(
        f"ℹ️ Target table already exists: {TARGET_TABLE}"
    )

    print(
        "The existing table will be replaced."
    )

else:

    print(
        f"Target table does not exist: {TARGET_TABLE}"
    )

    print(
        "A new Silver table will be created."
    )


# ============================================================
# 12. WRITE TO SILVER DELTA TABLE
# ============================================================

print("\n==============================================")
print("WRITING DATA TO SILVER")
print("==============================================")


print(
    "Target table:",
    TARGET_TABLE
)

print(
    "Target S3 path:",
    SILVER_PATH
)


(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .option(
        "path",
        SILVER_PATH
    )
    .saveAsTable(
        TARGET_TABLE
    )
)


print("\n✅ Silver table written successfully!")


# ============================================================
# 13. VALIDATE SILVER TABLE
# ============================================================

print("\n==============================================")
print("VALIDATING SILVER TABLE")
print("==============================================")


silver_count = (
    spark.table(TARGET_TABLE)
    .count()
)


print(
    "Bronze Count              :",
    bronze_count
)

print(
    "After Quality Filter      :",
    after_quality_count
)

print(
    "Duplicates Removed        :",
    duplicates_removed
)

print(
    "Final Silver Count        :",
    silver_count
)


# ============================================================
# 14. CHECK SILVER TABLE LOCATION
# ============================================================

print("\n==============================================")
print("SILVER TABLE STORAGE LOCATION")
print("==============================================")


silver_detail = spark.sql(
    f"""
    DESCRIBE DETAIL {TARGET_TABLE}
    """
)


display(
    silver_detail.select(
        "format",
        "location",
        "numFiles",
        "sizeInBytes"
    )
)


# ============================================================
# 15. VERIFY FINAL SILVER DATA
# ============================================================

print("\n==============================================")
print("FINAL SILVER DATA")
print("==============================================")


display(
    spark.sql(
        f"""
        SELECT
            Customer_ID,
            Customer_Name,
            Gender,
            Age,
            State,
            City,
            Bank_Branch,
            Account_Type,
            Transaction_ID,
            Transaction_Date,
            Transaction_Time,
            Transaction_Amount,
            Merchant_ID,
            Transaction_Type,
            Merchant_Category,
            Account_Balance,
            Transaction_Device,
            Transaction_Location,
            District,
            Device_Type,
            Is_Fraud,
            Transaction_Currency,
            Customer_Contact,
            Transaction_Description,
            Customer_Email,
            _silver_processed_timestamp,
            _source_layer
        FROM {TARGET_TABLE}
        LIMIT 20
        """
    )
)


# ============================================================
# 16. FRAUD RECORD VALIDATION
# ============================================================

print("\n==============================================")
print("FRAUD RECORD VALIDATION")
print("==============================================")


fraud_summary = spark.sql(
    f"""
    SELECT
        Is_Fraud,
        COUNT(*) AS Record_Count
    FROM {TARGET_TABLE}
    GROUP BY Is_Fraud
    ORDER BY Is_Fraud
    """
)


display(
    fraud_summary
)


# ============================================================
# 17. DATA QUALITY SUMMARY
# ============================================================

print("\n==============================================")
print("DATA QUALITY SUMMARY")
print("==============================================")


null_summary = spark.sql(
    f"""
    SELECT

        SUM(
            CASE
                WHEN Customer_ID IS NULL
                THEN 1 ELSE 0
            END
        ) AS Null_Customer_ID,

        SUM(
            CASE
                WHEN Transaction_ID IS NULL
                THEN 1 ELSE 0
            END
        ) AS Null_Transaction_ID,

        SUM(
            CASE
                WHEN Transaction_Date IS NULL
                THEN 1 ELSE 0
            END
        ) AS Null_Transaction_Date,

        SUM(
            CASE
                WHEN Transaction_Amount IS NULL
                THEN 1 ELSE 0
            END
        ) AS Null_Transaction_Amount,

        SUM(
            CASE
                WHEN Is_Fraud IS NULL
                THEN 1 ELSE 0
            END
        ) AS Null_Is_Fraud

    FROM {TARGET_TABLE}
    """
)


display(
    null_summary
)


# ============================================================
# 18. CHECK RESCUED DATA
# ============================================================

print("\n==============================================")
print("RESCUED DATA CHECK")
print("==============================================")


target_columns = (
    spark.table(TARGET_TABLE)
    .columns
)


if "_rescued_data" in target_columns:

    rescued_count = (
        spark.table(TARGET_TABLE)
        .filter(
            F.col("_rescued_data").isNotNull()
        )
        .count()
    )

    if rescued_count > 0:

        print(
            f"⚠️ Rescued records: {rescued_count}"
        )

        print(
            "Please review the _rescued_data column."
        )

    else:

        print(
            "✅ No rescued records found."
        )

else:

    print(
        "ℹ️ _rescued_data column not available."
    )


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("BRONZE -> SILVER TRANSFORMATION COMPLETED")
print("=" * 60)

print(
    "Source Table     :",
    SOURCE_TABLE
)

print(
    "Target Table     :",
    TARGET_TABLE
)

print(
    "Silver S3 Path   :",
    SILVER_PATH
)

print(
    "Bronze Count     :",
    bronze_count
)

print(
    "Quality Count    :",
    after_quality_count
)

print(
    "Duplicates Removed:",
    duplicates_removed
)

print(
    "Silver Count     :",
    silver_count
)

print("=" * 60)
print("✅ BRONZE -> SILVER COMPLETED SUCCESSFULLY")
print("=" * 60)